### This notebook will:
##### Call Alpha Vantage API
##### Pull stock data
##### Save raw JSON into Volume
##### One file per symbol per run

In [0]:
# Creating volume 
spark.sql("CREATE SCHEMA IF NOT EXISTS pyspark_module.stock_dlt")

spark.sql("""
CREATE VOLUME IF NOT EXISTS pyspark_module.stock_dlt.raw_stock_data
""")

DataFrame[]

In [0]:
# imports
import requests
import json
import time
from datetime import datetime, timezone

In [0]:
SYMBOLS = ["AAPL", "MSFT", "GOOGL", "TSLA"]
BASE_URL = "https://www.alphavantage.co/query"

RAW_BASE_PATH = "/Volumes/pyspark_module/stock_dlt/raw_stock_data"
# Retrieve Alpha Vantage API key securely from Databricks Secrets scope
API_KEY = dbutils.secrets.get(
    scope="alpha-vantage-scope",
    key="api-key"
)

#### Helper function to call Alpha Vantage API for a given stock symbol and endpoint. It builds the request parameters, sends the API request, checks for errors or rate-limit messages, and returns the API response as a Python dictionary.

In [0]:
def fetch_alpha_vantage(function_name, symbol):
    params = {
        "function": function_name,
        "symbol": symbol,
        "apikey": API_KEY
    }

    if function_name == "TIME_SERIES_DAILY":
        params["outputsize"] = "compact"

    response = requests.get(BASE_URL, params=params, timeout=30)
    data = response.json()

    if "Error Message" in data:
        raise Exception(f"API error for {symbol}, {function_name}: {data}")

    if "Information" in data:
        raise Exception(f"API limit or info message for {symbol}, {function_name}: {data}")

    return data

#### Fetch daily prices, latest quote, and company info for each symbol.Save each raw API response as JSON in the Volume for the DLT pipeline to read later.Sleep between symbols to reduce API rate-limit issues.

In [0]:
run_ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

for symbol in SYMBOLS:
    print(f"Processing {symbol}")

    daily_data = fetch_alpha_vantage("TIME_SERIES_DAILY", symbol)
    time.sleep(15)

    quote_data = fetch_alpha_vantage("GLOBAL_QUOTE", symbol)
    time.sleep(15)

    company_data = fetch_alpha_vantage("OVERVIEW", symbol)
    time.sleep(15)

    daily_path = f"{RAW_BASE_PATH}/daily_prices/symbol={symbol}/daily_{run_ts}.json"
    quote_path = f"{RAW_BASE_PATH}/quotes/symbol={symbol}/quote_{run_ts}.json"
    company_path = f"{RAW_BASE_PATH}/company_info/symbol={symbol}/company_{run_ts}.json"

    dbutils.fs.put(daily_path, json.dumps(daily_data), overwrite=True)
    dbutils.fs.put(quote_path, json.dumps(quote_data), overwrite=True)
    dbutils.fs.put(company_path, json.dumps(company_data), overwrite=True)

    print(f"Saved daily data: {daily_path}")
    print(f"Saved quote data: {quote_path}")
    print(f"Saved company info: {company_path}")

    time.sleep(15)

Processing AAPL
Wrote 13527 bytes.
Wrote 295 bytes.
Wrote 2078 bytes.
Saved daily data: /Volumes/pyspark_module/stock_dlt/raw_stock_data/daily_prices/symbol=AAPL/daily_20260428_095306.json
Saved quote data: /Volumes/pyspark_module/stock_dlt/raw_stock_data/quotes/symbol=AAPL/quote_20260428_095306.json
Saved company info: /Volumes/pyspark_module/stock_dlt/raw_stock_data/company_info/symbol=AAPL/company_20260428_095306.json
Processing MSFT
Wrote 13525 bytes.
Wrote 293 bytes.
Wrote 2352 bytes.
Saved daily data: /Volumes/pyspark_module/stock_dlt/raw_stock_data/daily_prices/symbol=MSFT/daily_20260428_095306.json
Saved quote data: /Volumes/pyspark_module/stock_dlt/raw_stock_data/quotes/symbol=MSFT/quote_20260428_095306.json
Saved company info: /Volumes/pyspark_module/stock_dlt/raw_stock_data/company_info/symbol=MSFT/company_20260428_095306.json
Processing GOOGL
Wrote 13527 bytes.
Wrote 294 bytes.
Wrote 2068 bytes.
Saved daily data: /Volumes/pyspark_module/stock_dlt/raw_stock_data/daily_prices